<a href="https://colab.research.google.com/github/AkshayJKulkarni/Mobility-Supply-Analytics/blob/main/Captain_Acquisition_and_Supply.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Rapido Data Science Take-Home Exercise
## Captain Acquisition, Supply & Airport Marketplace Analysis

### Objective

Analyze the captain onboarding funnel and airport marketplace data to:

1. Build the signup → approval funnel and identify the largest actionable leak.
2. Evaluate the effectiveness of `CAMP_WA_002` and assess whether increased investment is justified.
3. Characterize airport demand-supply mismatch and post-airport trip outcomes.
4. Recommend prioritized actions with quantified impact, risks, and measurement plans.

### Analysis Principles

- Define metrics, denominators, and cohorts before calculating results.
- Use mature cohorts where sufficient observation time is required.
- Distinguish descriptive association from causal impact.
- Quantify both percentages and absolute volume lost.
- Explicitly flag data-quality limitations and assumptions.
- Keep important findings reproducible from the raw CSV files.

In [ ]:
# Core libraries
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt

# Display settings
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

print("Libraries imported successfully.")

Libraries imported successfully.


In [ ]:
import pandas as pd

github_username = "AkshayJKulkarni"
repo_name = "Mobility-Supply-Analytics"

base_url = (
    f"https://raw.githubusercontent.com/"
    f"{github_username}/{repo_name}/main/"
)

file_names = [
    "captains",
    "doc_events",
    "approvals",
    "activation",
    "nudges",
    "airport_hourly",
    "airport_trips"
]

datasets = {}

print("Loading supply datasets from GitHub...\n")

for name in file_names:
    url = f"{base_url}{name}.csv"
    datasets[name] = pd.read_csv(url)
    print(f"✓ {name:20s} {datasets[name].shape}")

captains = datasets["captains"]
doc_events = datasets["doc_events"]
approvals = datasets["approvals"]
activation = datasets["activation"]
nudges = datasets["nudges"]
airport_hourly = datasets["airport_hourly"]
airport_trips = datasets["airport_trips"]

print("\nAll datasets loaded successfully.")

Loading supply datasets from GitHub...

✓ captains             (25000, 9)
✓ doc_events           (186282, 7)
✓ approvals            (25000, 5)
✓ activation           (4206, 5)
✓ nudges               (16314, 6)
✓ airport_hourly       (10248, 9)
✓ airport_trips        (60000, 9)

All datasets loaded successfully.


In [ ]:
datasets = {
    "captains": captains,
    "doc_events": doc_events,
    "approvals": approvals,
    "activation": activation,
    "nudges": nudges,
    "airport_hourly": airport_hourly,
    "airport_trips": airport_trips
}

for name, df in datasets.items():
    print(f"{name:16s}: {df.shape}")

captains        : (25000, 9)
doc_events      : (186282, 7)
approvals       : (25000, 5)
activation      : (4206, 5)
nudges          : (16314, 6)
airport_hourly  : (10248, 9)
airport_trips   : (60000, 9)


## 3. Data Quality Audit

Before calculating funnel or marketplace metrics, I first validate:

- Dataset sizes and schemas
- Duplicate records and identifiers
- Missing values
- Timestamp coverage
- Logical consistency between related tables

Missing values are not automatically treated as errors; their meaning depends on the field and business process.

In [ ]:
# Dataset-level audit

audit = []

for name, df in datasets.items():
    audit.append({
        "dataset": name,
        "rows": len(df),
        "columns": df.shape[1],
        "duplicate_rows": df.duplicated().sum(),
        "missing_cells": df.isna().sum().sum(),
        "columns_with_missing": (df.isna().sum() > 0).sum()
    })

audit_df = pd.DataFrame(audit)

audit_df

,dataset,rows,columns,duplicate_rows,missing_cells,columns_with_missing
0,captains,25000,9,0,0,0
1,doc_events,186282,7,0,166510,1
2,approvals,25000,5,0,24752,2
3,activation,4206,5,0,1957,4
4,nudges,16314,6,0,0,0
5,airport_hourly,10248,9,0,0,0
6,airport_trips,60000,9,0,0,0


In [ ]:
# Missing values by column

for name, df in datasets.items():
    missing = df.isna().sum()
    missing = missing[missing > 0]

    print(f"\n{name}")

    if len(missing) == 0:
        print("No missing values.")
    else:
        display(missing.to_frame("missing_count"))


captains
No missing values.

doc_events


,missing_count
failure_reason,166510



approvals


,missing_count
decision_ts,20359
last_stage_reached,4393



activation


,missing_count
first_order_ts,102
orders_d7,187
orders_d30,834
online_hours_d30,834



nudges
No missing values.

airport_hourly
No missing values.

airport_trips
No missing values.


In [ ]:
#schema-representation
for name, df in datasets.items():
    print(f"\n{'='*60}")
    print(name.upper())
    print(f"{'='*60}")
    display(
        pd.DataFrame({
            "column": df.columns,
            "dtype": df.dtypes.astype(str)
        })
    )


CAPTAINS


,column,dtype
captain_id,captain_id,object
signup_ts,signup_ts,object
city,city,object
vehicle_type,vehicle_type,object
acquisition_channel,acquisition_channel,object
signup_zone_id,signup_zone_id,object
device_tier,device_tier,object
app_language,app_language,object
age_band,age_band,object



DOC_EVENTS


,column,dtype
event_id,event_id,object
captain_id,captain_id,object
doc_type,doc_type,object
attempt_no,attempt_no,int64
event_type,event_type,object
event_ts,event_ts,object
failure_reason,failure_reason,object



APPROVALS


...

,column,dtype
captain_id,captain_id,object
campaign_id,campaign_id,object
channel,channel,object
sent_ts,sent_ts,object
delivered,delivered,int64
clicked,clicked,int64



AIRPORT_HOURLY


,column,dtype
zone_id,zone_id,object
zone_type,zone_type,object
hour_ts,hour_ts,object
requests,requests,int64
fulfilled_requests,fulfilled_requests,int64
unfulfilled_requests,unfulfilled_requests,int64
online_captains,online_captains,int64
avg_eta_min,avg_eta_min,float64
avg_surge_multiplier,avg_surge_multiplier,float64



AIRPORT_TRIPS


,column,dtype
trip_id,trip_id,object
pickup_zone_id,pickup_zone_id,object
drop_zone_id,drop_zone_id,object
drop_zone_type,drop_zone_type,object
request_ts,request_ts,object
trip_distance_km,trip_distance_km,float64
captain_cancelled,captain_cancelled,int64
got_return_fare_within_20min,got_return_fare_within_20min,int64
fare_inr,fare_inr,float64


## 4. Timestamp Standardization & Observation Window

The exercise states that the data was extracted at **2026-06-30 23:59 IST**.

For onboarding analysis, recent signups should not automatically be considered dropouts because they may not have had enough time to complete the document process.

Therefore, I use a **30-day mature signup cohort** for the main funnel analysis.

In [ ]:
# Timestamp standardization
# Raw timestamps use a day-first format: DD-MM-YYYY HH:MM
# Timestamp Standardization & Observation Window


timestamp_columns = {
    "captains": ["signup_ts"],
    "doc_events": ["event_ts"],
    "approvals": ["decision_ts"],
    "activation": ["first_order_ts"],
    "nudges": ["sent_ts"],
    "airport_hourly": ["hour_ts"],
    "airport_trips": ["request_ts"]
}

# Standardize timestamps
for name, columns in timestamp_columns.items():
    for col in columns:

        if name == "captains" and col == "signup_ts":
            # Raw format confirmed from data:
            # DD-MM-YYYY HH:MM
            datasets[name][col] = pd.to_datetime(
                datasets[name][col],
                format="%d-%m-%Y %H:%M",
                errors="coerce"
            )

        else:
            # Other timestamp columns are already in a
            # directly parseable datetime format.
            datasets[name][col] = pd.to_datetime(
                datasets[name][col],
                errors="coerce"
            )


# Reassign standardized datasets
captains = datasets["captains"]
doc_events = datasets["doc_events"]
approvals = datasets["approvals"]
activation = datasets["activation"]
nudges = datasets["nudges"]
airport_hourly = datasets["airport_hourly"]
airport_trips = datasets["airport_trips"]


# Dataset extraction cutoff
EXTRACTION_CUTOFF = pd.Timestamp("2026-06-30 23:59:00")

# 30-day maturity cutoff
MATURE_CUTOFF = EXTRACTION_CUTOFF - pd.Timedelta(days=30)


print("Extraction cutoff:", EXTRACTION_CUTOFF)
print("Maturity cutoff:", MATURE_CUTOFF)

Extraction cutoff: 2026-06-30 23:59:00
Maturity cutoff: 2026-05-31 23:59:00


TimeStamp Validation

In [ ]:
#Timestamp & Dataset Validation

print("=" * 70)
print("TIMESTAMP & DATASET VALIDATION")
print("=" * 70)


# ------------------------------------------------------------
# 1. CAPTAINS — SIGNUP TIMESTAMP VALIDATION
# ------------------------------------------------------------

print("\n1. CAPTAINS — signup_ts")
print("-" * 70)

print("Dtype:", captains["signup_ts"].dtype)
print("Minimum:", captains["signup_ts"].min())
print("Maximum:", captains["signup_ts"].max())
print("Missing:", captains["signup_ts"].isna().sum())
print("Total:", len(captains))

print("\nSignup month distribution:")

signup_months = (
    captains["signup_ts"]
    .dropna()
    .dt.to_period("M")
    .value_counts()
    .sort_index()
)

display(signup_months.to_frame("signups"))


# ------------------------------------------------------------
# 2. SIGNUPS AFTER EXTRACTION CUTOFF
# ------------------------------------------------------------

print("\n2. EXTRACTION CUTOFF CHECK")
print("-" * 70)

print("Extraction cutoff:", EXTRACTION_CUTOFF)

signup_after_cutoff = captains[
    captains["signup_ts"] > EXTRACTION_CUTOFF
]

print("Signups after extraction cutoff:", len(signup_after_cutoff))

if len(signup_after_cutoff) == 0:
    print("✓ No signup timestamps occur after the extraction cutoff.")
else:
    print("⚠ Signup timestamps occur after the extraction cutoff.")


# ------------------------------------------------------------
# 3. ALL TIMESTAMP COLUMNS
# ------------------------------------------------------------

print("\n3. ALL TIMESTAMP COLUMNS")
print("-" * 70)

for name, columns in timestamp_columns.items():

    df = datasets[name]

    for col in columns:

        print(f"\n{name}.{col}")
        print("  Dtype:   ", df[col].dtype)
        print("  Minimum: ", df[col].min())
        print("  Maximum: ", df[col].max())
        print("  Missing: ", df[col].isna().sum())
        print("  Total:   ", len(df))


# ------------------------------------------------------------
# 4. RAW DATASET SIZE / SHAPE CHECK
# ------------------------------------------------------------

print("\n4. DATASET SIZE CHECK")
print("-" * 70)

expected_shapes = {
    "captains": (25000, 9),
    "doc_events": (186282, 7),
    "approvals": (25000, 5),
    "activation": (4206, 5),
    "nudges": (16314, 6),
    "airport_hourly": (10248, 9),
    "airport_trips": (60000, 9)
}

shape_results = []

for name, expected_shape in expected_shapes.items():

    actual_shape = datasets[name].shape

    status = "✓" if actual_shape == expected_shape else "⚠"

    print(
        f"{status} {name:20s} "
        f"Actual: {actual_shape} | "
        f"Expected: {expected_shape}"
    )

    shape_results.append({
        "dataset": name,
        "actual_rows": actual_shape[0],
        "actual_columns": actual_shape[1],
        "expected_rows": expected_shape[0],
        "expected_columns": expected_shape[1],
        "status": status
    })


# ------------------------------------------------------------
# 5. TIMESTAMP DTYPE CHECK
# ------------------------------------------------------------

print("\n5. TIMESTAMP DTYPE CHECK")
print("-" * 70)

for name, columns in timestamp_columns.items():

    for col in columns:

        dtype = datasets[name][col].dtype

        if pd.api.types.is_datetime64_any_dtype(dtype):
            status = "✓"
        else:
            status = "⚠"

        print(
            f"{status} {name}.{col}: {dtype}"
        )


# ------------------------------------------------------------
# 6. TIMESTAMP MISSING-VALUE SUMMARY
# ------------------------------------------------------------

print("\n6. TIMESTAMP MISSING-VALUE SUMMARY")
print("-" * 70)

timestamp_missing = []

for name, columns in timestamp_columns.items():

    for col in columns:

        df = datasets[name]

        missing = df[col].isna().sum()
        total = len(df)
        percentage = (missing / total) * 100

        timestamp_missing.append({
            "dataset": name,
            "column": col,
            "missing": missing,
            "total": total,
            "missing_pct": round(percentage, 2)
        })

display(pd.DataFrame(timestamp_missing))


# ------------------------------------------------------------
# 7. AIRPORT DATA DATE-RANGE CHECKS
# ------------------------------------------------------------

print("\n7. AIRPORT DATA DATE-RANGE CHECK")
print("-" * 70)

print(
    "airport_hourly:",
    airport_hourly["hour_ts"].min(),
    "to",
    airport_hourly["hour_ts"].max()
)

print(
    "airport_trips:",
    airport_trips["request_ts"].min(),
    "to",
    airport_trips["request_ts"].max()
)


# ------------------------------------------------------------
# 8. FINAL VALIDATION SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("FINAL VALIDATION SUMMARY")
print("=" * 70)

signup_dtype_valid = pd.api.types.is_datetime64_any_dtype(
    captains["signup_ts"]
)

signup_missing = captains["signup_ts"].isna().sum()

signup_cutoff_violations = len(signup_after_cutoff)

shape_check_passed = all(
    datasets[name].shape == expected_shape
    for name, expected_shape in expected_shapes.items()
)

print("\nSignup timestamp dtype valid:",
      "YES ✓" if signup_dtype_valid else "NO ⚠")

print("Missing signup timestamps:",
      signup_missing)

print("Signups after extraction cutoff:",
      signup_cutoff_violations)

print("All dataset shapes correct:",
      "YES ✓" if shape_check_passed else "NO ⚠")

print("\nMaturity cutoff:", MATURE_CUTOFF)

if (
    signup_dtype_valid
    and signup_missing == 0
    and signup_cutoff_violations == 0
    and shape_check_passed
):
    print("\n✓ BASIC DATA/TIMESTAMP VALIDATION PASSED")
    print("Safe to proceed to cohort definition.")
else:
    print("\n⚠ VALIDATION FAILED")
    print("Do NOT proceed to funnel analysis yet.")

TIMESTAMP & DATASET VALIDATION

1. CAPTAINS — signup_ts
----------------------------------------------------------------------
Dtype: datetime64[ns]
Minimum: 2026-01-01 06:08:00
Maximum: 2026-06-30 21:57:00
Missing: 0
Total: 25000

Signup month distribution:


,signups
signup_ts,
2026-01,3716
2026-02,3600
2026-03,4188
2026-04,4123
2026-05,4580
2026-06,4793



2. EXTRACTION CUTOFF CHECK
----------------------------------------------------------------------
Extraction cutoff: 2026-06-30 23:59:00
Signups after extraction cutoff: 0
✓ No signup timestamps occur after the extraction cutoff.

3. ALL TIMESTAMP COLUMNS
----------------------------------------------------------------------

captains.signup_ts
  Dtype:    datetime64[ns]
  Minimum:  2026-01-01 06:08:00
  Maximum:  2026-06-30 21:57:00
  Missing:  0
  Total:    25000

doc_events.event_ts
  Dtype:    datetime64[ns]
  Minimum:  2026-01-01 13:30:50.795752785
  Maximum:  2026-06-30 23:56:42.723769018
  Missing:  0
  Total:    186282

approvals.decision_ts
  Dtype:    datetime64[ns]
  Minimum:  2026-01-05 21:40:24.347261353
  Maximum:  2026-06-30 22:26:04.692838376
  Missing:  20359
  Total:    25000

activation.first_order_ts
  Dtype:    datetime64[ns]
  Minimum:  2026-01-06 11:57:58.804450784
  Maximum:  2026-06-30 21:36:06.089121524
  Missing:  102
  Total:    4206

nudges.sent_ts
  Dtype

,dataset,column,missing,total,missing_pct
0,captains,signup_ts,0,25000,0.000
1,doc_events,event_ts,0,186282,0.000
2,approvals,decision_ts,20359,25000,81.440
3,activation,first_order_ts,102,4206,2.430
4,nudges,sent_ts,0,16314,0.000
5,airport_hourly,hour_ts,0,10248,0.000
6,airport_trips,request_ts,0,60000,0.000



7. AIRPORT DATA DATE-RANGE CHECK
----------------------------------------------------------------------
airport_hourly: 2026-05-01 00:00:00 to 2026-06-30 23:00:00
airport_trips: 2026-05-01 00:00:00 to 2026-06-30 23:00:00

FINAL VALIDATION SUMMARY

Signup timestamp dtype valid: YES ✓
Missing signup timestamps: 0
Signups after extraction cutoff: 0
All dataset shapes correct: YES ✓

Maturity cutoff: 2026-05-31 23:59:00

✓ BASIC DATA/TIMESTAMP VALIDATION PASSED
Safe to proceed to cohort definition.


## Cohort Definition

### Mature Signup Cohort

The primary funnel analysis uses a mature signup cohort to ensure captains have sufficient observation time before the dataset extraction cutoff.

**Extraction cutoff:** 30 June 2026, 23:59 IST

**Observation window:** 30 days

**Maturity cutoff:** 31 May 2026, 23:59 IST

Captains who signed up on or before the maturity cutoff are included in the mature cohort.

This avoids undercounting downstream outcomes for recent June signups that may not yet have had 30 days to complete onboarding and activation.

For the signup → approval funnel, the denominator is the number of captains in this mature signup cohort.

**Assumption:** A 30-day observation window is used because the dataset explicitly provides 30-day activation measures (`orders_d30` and `online_hours_d30`) and the extraction cutoff is 30 June 2026.

**What would change the answer:** If the business defines a different operational SLA or observation window for onboarding completion, the cohort definition and resulting funnel metrics would need to be recalculated.

In [ ]:
# Create Mature Signup Cohort

mature_captains = captains[
    captains["signup_ts"] <= MATURE_CUTOFF
].copy()

print("=" * 70)
print("MATURE SIGNUP COHORT")
print("=" * 70)

print("\nExtraction cutoff:", EXTRACTION_CUTOFF)
print("Maturity cutoff:", MATURE_CUTOFF)

print("\nFull signup population:", len(captains))
print("Mature signup cohort:", len(mature_captains))

print("\nMature cohort percentage:",
      round(len(mature_captains) / len(captains) * 100, 2), "%")


# Monthly mature cohort distribution
mature_monthly = (
    mature_captains
    .assign(
        signup_month=mature_captains["signup_ts"].dt.to_period("M")
    )
    .groupby("signup_month")
    .size()
    .rename("signups")
    .reset_index()
)

print("\nMature cohort by signup month:")
display(mature_monthly)

MATURE SIGNUP COHORT

Extraction cutoff: 2026-06-30 23:59:00
Maturity cutoff: 2026-05-31 23:59:00

Full signup population: 25000
Mature signup cohort: 20207

Mature cohort percentage: 80.83 %

Mature cohort by signup month:


,signup_month,signups
0,2026-01,3716
1,2026-02,3600
2,2026-03,4188
3,2026-04,4123
4,2026-05,4580


## Part A1 — Signup → Approved Funnel

### Objective

Build a stage-by-stage onboarding funnel for the mature signup cohort and quantify both conversion and absolute volume lost at each stage.

### Cohort

Only captains who signed up on or before 31 May 2026 are included, giving each captain at least 30 days of observation before the 30 June 2026 extraction cutoff.

### Funnel stages

For Auto/Cab:

Signup → DL → RC → Aadhaar → Permit → Fitness → Insurance → Approved

For E-Rickshaw:

Signup → DL → RC → Aadhaar → Fitness → Insurance → Approved

Permit is not required for E-Rickshaw.

### Stage definition

A captain is considered to have passed a document stage if they have at least one successful (`pass`) document event for that required document.

A captain is considered approved if their final approval status is `approved`.

For each stage, report:

- Captains reaching the stage
- Conversion rate from the previous stage
- Cumulative conversion from signup
- Absolute volume lost from the previous stage
- Absolute volume lost from signup

The denominator for the overall funnel is the mature signup cohort of 20,207 captains.

### Important validation

The document sequence and vehicle-specific requirements will be validated against the raw event data before calculating the final funnel.

In [ ]:
# Inspect Document Event Structure

print("=" * 70)
print("DOCUMENT EVENT STRUCTURE")
print("=" * 70)

print("\nDocument types:")
display(
    doc_events["doc_type"]
    .value_counts(dropna=False)
    .to_frame("count")
)

print("\nEvent types:")
display(
    doc_events["event_type"]
    .value_counts(dropna=False)
    .to_frame("count")
)

print("\nAttempt numbers:")
display(
    doc_events["attempt_no"]
    .value_counts(dropna=False)
    .sort_index()
    .to_frame("count")
)

print("\nFailure reasons:")
display(
    doc_events["failure_reason"]
    .value_counts(dropna=False)
    .to_frame("count")
)

DOCUMENT EVENT STRUCTURE

Document types:


,count
doc_type,
DL,50780
RC,48099
AADHAAR,31117
FITNESS,21884
PERMIT,20807
INSURANCE,13595



Event types:


,count
event_type,
upload_success,93236
verification_pass,73274
verification_fail,19772



Attempt numbers:


,count
attempt_no,
1,162176
2,21863
3,2243



Failure reasons:


,count
failure_reason,
NaN,166510
image_blurred,4767
ocr_low_confidence,3696
name_mismatch,3173
document_expired,2686
details_not_legible,2171
wrong_document_type,2083
duplicate_document,1196


In [ ]:
# Build Document Verification-Pass Flags

print("=" * 70)
print("DOCUMENT PASS FLAGS")
print("=" * 70)

# Work only with the mature signup cohort
mature_ids = mature_captains["captain_id"]

mature_doc_events = doc_events[
    doc_events["captain_id"].isin(mature_ids)
].copy()

print("Document events for mature cohort:", len(mature_doc_events))

# A document is considered cleared when it has at least one
# successful verification_pass event.
passed_docs = (
    mature_doc_events[
        mature_doc_events["event_type"] == "verification_pass"
    ]
    .groupby(["captain_id", "doc_type"])
    .size()
    .unstack(fill_value=0)
)

# Convert counts into boolean pass flags
for doc in ["DL", "RC", "AADHAAR", "PERMIT", "FITNESS", "INSURANCE"]:
    if doc in passed_docs.columns:
        passed_docs[f"{doc}_passed"] = passed_docs[doc] > 0
    else:
        passed_docs[f"{doc}_passed"] = False


# Keep only the boolean columns
doc_pass_flags = passed_docs[
    [
        "DL_passed",
        "RC_passed",
        "AADHAAR_passed",
        "PERMIT_passed",
        "FITNESS_passed",
        "INSURANCE_passed"
    ]
].reset_index()


print("\nDocument pass flags created.")

display(doc_pass_flags.head(10))

DOCUMENT PASS FLAGS
Document events for mature cohort: 153120

Document pass flags created.


doc_type,captain_id,DL_passed,RC_passed,AADHAAR_passed,PERMIT_passed,FITNESS_passed,INSURANCE_passed
0,CPT100000,True,True,True,True,False,False
1,CPT100002,True,True,True,True,True,True
2,CPT100003,True,False,False,False,False,False
3,CPT100004,True,True,True,False,True,False
4,CPT100006,True,True,True,True,True,True
5,CPT100008,True,True,True,True,True,True
6,CPT100009,True,False,False,False,False,False
7,CPT100010,True,True,True,False,False,False
8,CPT100011,True,False,False,False,False,False
9,CPT100012,True,True,True,False,True,True


In [ ]:
# Merge Document Pass Flags With Mature Captains

funnel_base = mature_captains[
    [
        "captain_id",
        "signup_ts",
        "city",
        "vehicle_type",
        "acquisition_channel",
        "device_tier",
        "app_language",
        "age_band"
    ]
].copy()

funnel_base = funnel_base.merge(
    doc_pass_flags,
    on="captain_id",
    how="left"
)

# Captains with no document-pass events get False for all documents
document_flag_columns = [
    "DL_passed",
    "RC_passed",
    "AADHAAR_passed",
    "PERMIT_passed",
    "FITNESS_passed",
    "INSURANCE_passed"
]

funnel_base[document_flag_columns] = (
    funnel_base[document_flag_columns]
    .fillna(False)
    .astype(bool)
)

print("Funnel base shape:", funnel_base.shape)

display(funnel_base.head(10))

Funnel base shape: (20207, 14)


/tmp/ipykernel_1473/160535071.py:34: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .fillna(False)


,captain_id,signup_ts,city,vehicle_type,acquisition_channel,device_tier,app_language,age_band,DL_passed,RC_passed,AADHAAR_passed,PERMIT_passed,FITNESS_passed,INSURANCE_passed
0,CPT104359,2026-01-01 06:08:00,Pune,Auto,organic_app,low,kn,25-34,True,True,True,True,True,True
1,CPT120008,2026-01-01 06:52:00,Pune,Cab,referral,mid,mr,45+,True,True,False,False,False,False
2,CPT102049,2026-01-01 07:02:00,Hyderabad,Cab,organic_app,high,te,35-44,True,True,True,False,False,False
3,CPT123006,2026-01-01 07:15:00,Pune,Cab,organic_app,high,hi,25-34,True,True,True,True,True,False
4,CPT102642,2026-01-01 07:27:00,Bangalore,ERickshaw,gc_telecalling,low,hi,18-24,True,True,True,False,False,False
5,CPT105916,2026-01-01 07:32:00,Delhi,Cab,fos_field,high,hi,35-44,True,True,True,True,True,True
6,CPT123016,2026-01-01 08:02:00,Hyderabad,Cab,organic_app,low,hi,35-44,True,False,False,False,False,False
7,CPT103410,2026-01-01 08:20:00,Hyderabad,Auto,fos_field,mid,hi,45+,True,True,True,True,True,False
8,CPT107181,2026-01-01 08:24:00,Hyderabad,Auto,organic_app,low,en,25-34,True,True,True,True,True,True
9,CPT122385,2026-01-01 08:28:00,Hyderabad,Cab,paid_digital,low,en,25-34,True,True,True,True,False,False


In [ ]:
# Construct Sequential Funnel Stage

print("=" * 70)
print("SEQUENTIAL FUNNEL STAGE")
print("=" * 70)


def get_funnel_stage(row):
    """
    Determine the furthest consecutive document stage reached.

    Auto/Cab:
    Signup → DL → RC → Aadhaar → Permit → Fitness → Insurance

    E-Rickshaw:
    Signup → DL → RC → Aadhaar → Fitness → Insurance

    A stage is counted only if all required previous stages
    have also been cleared.
    """

    vehicle = row["vehicle_type"]

    # Start at signup
    if not row["DL_passed"]:
        return "Signup"

    if not row["RC_passed"]:
        return "DL"

    if not row["AADHAAR_passed"]:
        return "RC"

    # Permit is required only for Auto/Cab
    if vehicle in ["Auto", "Cab"]:

        if not row["PERMIT_passed"]:
            return "Aadhaar"

        if not row["FITNESS_passed"]:
            return "Permit"

    else:
        # E-Rickshaw skips Permit
        if not row["FITNESS_passed"]:
            return "Aadhaar"

    if not row["INSURANCE_passed"]:
        return "Fitness"

    return "Insurance"


funnel_base["furthest_stage_reached"] = (
    funnel_base.apply(get_funnel_stage, axis=1)
)

print("\nFurthest-stage distribution:")

stage_distribution = (
    funnel_base["furthest_stage_reached"]
    .value_counts()
)

display(stage_distribution.to_frame("captains"))

SEQUENTIAL FUNNEL STAGE

Furthest-stage distribution:


,captains
furthest_stage_reached,
DL,4887
Insurance,3888
Fitness,2979
Aadhaar,2888
Signup,2312
Permit,1876
RC,1377


In [ ]:
# Validate Approval Outcomes

print("=" * 70)
print("APPROVAL STATUS VALIDATION")
print("=" * 70)

print("\nFinal status distribution — full dataset:")
display(
    approvals["final_status"]
    .value_counts(dropna=False)
    .to_frame("captains")
)

print("\nLast stage reached distribution — full dataset:")
display(
    approvals["last_stage_reached"]
    .value_counts(dropna=False)
    .to_frame("captains")
)

print("\nDocs cleared distribution:")
display(
    approvals["docs_cleared"]
    .value_counts(dropna=False)
    .sort_index()
    .to_frame("captains")
)

APPROVAL STATUS VALIDATION

Final status distribution — full dataset:


,captains
final_status,
dropped_in_docs,19062
approved,4206
in_progress,1297
rejected,435



Last stage reached distribution — full dataset:


,captains
last_stage_reached,
RC,6031
NaN,4393
PERMIT,3609
FITNESS,3126
DL,2892
INSURANCE,2801
AADHAAR,1713
background_check_failed,113
vehicle_age_policy,111



Docs cleared distribution:


,captains
docs_cleared,
0,2892
1,6031
2,1713
3,3609
4,3126
5,4027
6,3602


In [ ]:
# Reconcile Document Events With Approval Data

print("=" * 70)
print("MATURE COHORT — APPROVAL RECONCILIATION")
print("=" * 70)

# Keep approval records only for the mature signup cohort
mature_approvals = approvals[
    approvals["captain_id"].isin(mature_ids)
].copy()

print("\nMature cohort size:", len(mature_captains))
print("Mature approval records:", len(mature_approvals))

# ------------------------------------------------------------
# 1. Final approval status within mature cohort
# ------------------------------------------------------------

print("\n1. FINAL STATUS — MATURE COHORT")
print("-" * 70)

display(
    mature_approvals["final_status"]
    .value_counts(dropna=False)
    .to_frame("captains")
)


# ------------------------------------------------------------
# 2. Docs cleared within mature cohort
# ------------------------------------------------------------

print("\n2. DOCS CLEARED — MATURE COHORT")
print("-" * 70)

display(
    mature_approvals["docs_cleared"]
    .value_counts(dropna=False)
    .sort_index()
    .to_frame("captains")
)


# ------------------------------------------------------------
# 3. Compare event-derived document completion
#    with approval docs_cleared
# ------------------------------------------------------------

reconciliation = funnel_base.merge(
    mature_approvals[
        [
            "captain_id",
            "final_status",
            "last_stage_reached",
            "docs_cleared"
        ]
    ],
    on="captain_id",
    how="left"
)

print("\n3. RECONCILIATION SAMPLE")
print("-" * 70)

display(
    reconciliation[
        [
            "captain_id",
            "vehicle_type",
            "DL_passed",
            "RC_passed",
            "AADHAAR_passed",
            "PERMIT_passed",
            "FITNESS_passed",
            "INSURANCE_passed",
            "docs_cleared",
            "last_stage_reached",
            "final_status"
        ]
    ].head(15)
)


# ------------------------------------------------------------
# 4. Check missing approval records
# ------------------------------------------------------------

missing_approval_records = reconciliation["final_status"].isna().sum()

print("\n4. APPROVAL RECORD COVERAGE")
print("-" * 70)

print(
    "Mature captains without approval record:",
    missing_approval_records
)


# ------------------------------------------------------------
# 5. Approved captains vs full document completion
# ------------------------------------------------------------

approved = reconciliation[
    reconciliation["final_status"] == "approved"
].copy()

all_docs_passed = (
    approved["DL_passed"]
    & approved["RC_passed"]
    & approved["AADHAAR_passed"]
    & approved["FITNESS_passed"]
    & approved["INSURANCE_passed"]
    & (
        (approved["vehicle_type"] == "ERickshaw")
        | approved["PERMIT_passed"]
    )
)

print("\n5. APPROVED CAPTAINS WITH ALL REQUIRED DOCS PASSED")
print("-" * 70)

print("Approved captains:", len(approved))
print("Approved with all required docs passed:", all_docs_passed.sum())
print(
    "Approved without all required docs passed:",
    (~all_docs_passed).sum()
)

MATURE COHORT — APPROVAL RECONCILIATION

Mature cohort size: 20207
Mature approval records: 20207

1. FINAL STATUS — MATURE COHORT
----------------------------------------------------------------------


,captains
final_status,
dropped_in_docs,16319
approved,3532
rejected,356



2. DOCS CLEARED — MATURE COHORT
----------------------------------------------------------------------


,captains
docs_cleared,
0,2312
1,4887
2,1377
3,2888
4,2552
5,3311
6,2880



3. RECONCILIATION SAMPLE
----------------------------------------------------------------------


,captain_id,vehicle_type,DL_passed,RC_passed,AADHAAR_passed,PERMIT_passed,FITNESS_passed,INSURANCE_passed,docs_cleared,last_stage_reached,final_status
0,CPT104359,Auto,True,True,True,True,True,True,6,NaN,approved
1,CPT120008,Cab,True,True,False,False,False,False,2,AADHAAR,dropped_in_docs
2,CPT102049,Cab,True,True,True,False,False,False,3,PERMIT,dropped_in_docs
3,CPT123006,Cab,True,True,True,True,True,False,5,INSURANCE,dropped_in_docs
4,CPT102642,ERickshaw,True,True,True,False,False,False,3,PERMIT,dropped_in_docs
5,CPT105916,Cab,True,True,True,True,True,True,6,NaN,approved
6,CPT123016,Cab,True,False,False,False,False,False,1,RC,dropped_in_docs
7,CPT103410,Auto,True,True,True,True,True,False,5,INSURANCE,dropped_in_docs
8,CPT107181,Auto,True,True,True,True,True,True,6,background_check_failed,rejected
9,CPT122385,Cab,True,True,True,True,False,False,4,FITNESS,dropped_in_docs



4. APPROVAL RECORD COVERAGE
----------------------------------------------------------------------
Mature captains without approval record: 0

5. APPROVED CAPTAINS WITH ALL REQUIRED DOCS PASSED
----------------------------------------------------------------------
Approved captains: 3532
Approved with all required docs passed: 3532
Approved without all required docs passed: 0


In [ ]:
# Build Correct Vehicle-Specific A1 Funnel

print("=" * 80)
print("PART A1 — SIGNUP → APPROVED FUNNEL")
print("=" * 80)


# ------------------------------------------------------------
# 1. CUMULATIVE STAGE DEFINITIONS
# ------------------------------------------------------------
# A captain reaches a stage only if all required previous
# documents have been passed.
#
# Auto/Cab:
# Signup → DL → RC → Aadhaar → Permit → Fitness → Insurance
#
# E-Rickshaw:
# Signup → DL → RC → Aadhaar → Fitness → Insurance
# Permit is NOT required.


funnel_base["reached_DL"] = (
    funnel_base["DL_passed"]
)

funnel_base["reached_RC"] = (
    funnel_base["DL_passed"]
    & funnel_base["RC_passed"]
)

funnel_base["reached_Aadhaar"] = (
    funnel_base["DL_passed"]
    & funnel_base["RC_passed"]
    & funnel_base["AADHAAR_passed"]
)

funnel_base["reached_Permit"] = (
    funnel_base["reached_Aadhaar"]
    & (
        (funnel_base["vehicle_type"] == "ERickshaw")
        | funnel_base["PERMIT_passed"]
    )
)

funnel_base["reached_Fitness"] = (
    funnel_base["reached_Permit"]
    & funnel_base["FITNESS_passed"]
)

funnel_base["reached_Insurance"] = (
    funnel_base["reached_Fitness"]
    & funnel_base["INSURANCE_passed"]
)


# ------------------------------------------------------------
# 2. COUNT CAPTAINS AT EACH DOCUMENT STAGE
# ------------------------------------------------------------

signup_count = len(funnel_base)

dl_count = funnel_base["reached_DL"].sum()

rc_count = funnel_base["reached_RC"].sum()

aadhaar_count = funnel_base["reached_Aadhaar"].sum()

permit_count = funnel_base["reached_Permit"].sum()

fitness_count = funnel_base["reached_Fitness"].sum()

insurance_count = funnel_base["reached_Insurance"].sum()


# ------------------------------------------------------------
# 3. FINAL APPROVAL COUNT
# ------------------------------------------------------------

approved_count = (
    mature_approvals[
        mature_approvals["final_status"] == "approved"
    ]
    .shape[0]
)


# ------------------------------------------------------------
# 4. CREATE FUNNEL TABLE
# ------------------------------------------------------------

funnel_counts = pd.DataFrame({
    "stage": [
        "Signup",
        "DL",
        "RC",
        "Aadhaar",
        "Permit",
        "Fitness",
        "Insurance",
        "Approved"
    ],
    "captains": [
        signup_count,
        dl_count,
        rc_count,
        aadhaar_count,
        permit_count,
        fitness_count,
        insurance_count,
        approved_count
    ]
})


# ------------------------------------------------------------
# 5. FUNNEL METRICS
# ------------------------------------------------------------

funnel_counts["pct_of_signup"] = (
    funnel_counts["captains"]
    / signup_count
    * 100
)

funnel_counts["conversion_from_previous"] = (
    funnel_counts["captains"]
    / funnel_counts["captains"].shift(1)
    * 100
)

funnel_counts["volume_lost_from_previous"] = (
    funnel_counts["captains"].shift(1)
    - funnel_counts["captains"]
)

funnel_counts["volume_lost_from_signup"] = (
    signup_count
    - funnel_counts["captains"]
)


# Signup has no previous stage
funnel_counts.loc[
    0, "conversion_from_previous"
] = 100

funnel_counts.loc[
    0, "volume_lost_from_previous"
] = 0


# ------------------------------------------------------------
# 6. ROUND METRICS
# ------------------------------------------------------------

funnel_counts["pct_of_signup"] = (
    funnel_counts["pct_of_signup"].round(2)
)

funnel_counts["conversion_from_previous"] = (
    funnel_counts["conversion_from_previous"].round(2)
)


# ------------------------------------------------------------
# 7. DISPLAY
# ------------------------------------------------------------

print("\nFinal A1 Funnel:")
display(funnel_counts)

PART A1 — SIGNUP → APPROVED FUNNEL

Final A1 Funnel:


,stage,captains,pct_of_signup,conversion_from_previous,volume_lost_from_previous,volume_lost_from_signup
0,Signup,20207,100.000,100.000,0.000,0
1,DL,17895,88.560,88.560,"2,312.000",2312
2,RC,13008,64.370,72.690,"4,887.000",7199
3,Aadhaar,11631,57.560,89.410,"1,377.000",8576
4,Permit,9288,45.960,79.860,"2,343.000",10919
5,Fitness,6867,33.980,73.930,"2,421.000",13340
6,Insurance,3888,19.240,56.620,"2,979.000",16319
7,Approved,3532,17.480,90.840,356.000,16675


In [ ]:
# Define RC Dropout Population

print("=" * 80)
print("PART A2 — RC DROPOUT ANALYSIS")
print("=" * 80)

# Captains who cleared DL but did not clear RC
rc_dropouts = funnel_base[
    funnel_base["reached_DL"]
    & ~funnel_base["reached_RC"]
].copy()

# Captains who entered RC stage
rc_entrants = funnel_base[
    funnel_base["reached_DL"]
].copy()

print("\nRC entrants:", len(rc_entrants))
print("RC dropouts:", len(rc_dropouts))

print(
    "RC dropout rate:",
    round(len(rc_dropouts) / len(rc_entrants) * 100, 2),
    "%"
)

print(
    "RC completion rate:",
    round(
        len(funnel_base[funnel_base["reached_RC"]])
        / len(rc_entrants)
        * 100,
        2
    ),
    "%"
)

PART A2 — RC DROPOUT ANALYSIS

RC entrants: 17895
RC dropouts: 4887
RC dropout rate: 27.31 %
RC completion rate: 72.69 %


In [ ]:
# RC Failure Evidence

rc_failure_events = doc_events[
    (doc_events["captain_id"].isin(rc_dropouts["captain_id"]))
    & (doc_events["doc_type"] == "RC")
    & (doc_events["event_type"] == "verification_fail")
].copy()

rc_failure_captains = set(
    rc_failure_events["captain_id"]
)

rc_dropouts_with_failure = rc_dropouts[
    rc_dropouts["captain_id"].isin(rc_failure_captains)
]

rc_dropouts_without_failure = rc_dropouts[
    ~rc_dropouts["captain_id"].isin(rc_failure_captains)
]

print("=" * 80)
print("RC DROPOUT — FAILURE EVIDENCE")
print("=" * 80)

print("\nTotal RC dropouts:", len(rc_dropouts))

print(
    "RC dropouts with at least one verification failure:",
    len(rc_dropouts_with_failure)
)

print(
    "RC dropouts without a recorded verification failure:",
    len(rc_dropouts_without_failure)
)

print(
    "\n% with recorded RC verification failure:",
    round(
        len(rc_dropouts_with_failure)
        / len(rc_dropouts)
        * 100,
        2
    ),
    "%"
)

print(
    "% without recorded RC verification failure:",
    round(
        len(rc_dropouts_without_failure)
        / len(rc_dropouts)
        * 100,
        2
    ),
    "%"
)

RC DROPOUT — FAILURE EVIDENCE

Total RC dropouts: 4887
RC dropouts with at least one verification failure: 2727
RC dropouts without a recorded verification failure: 2160

% with recorded RC verification failure: 55.8 %
% without recorded RC verification failure: 44.2 %


In [ ]:
# RC Verification Failure Reasons

print("=" * 80)
print("RC VERIFICATION FAILURE REASONS")
print("=" * 80)

# Failure events among RC dropout captains
rc_failure_reasons = (
    rc_failure_events["failure_reason"]
    .value_counts(dropna=False)
    .rename_axis("failure_reason")
    .reset_index(name="failure_events")
)

# Percentage of all RC failure events
rc_failure_reasons["pct_of_failure_events"] = (
    rc_failure_reasons["failure_events"]
    / rc_failure_reasons["failure_events"].sum()
    * 100
)

rc_failure_reasons["pct_of_failure_events"] = (
    rc_failure_reasons["pct_of_failure_events"].round(2)
)

print("\nFailure events by reason:")
display(rc_failure_reasons)


# ------------------------------------------------------------
# Unique affected captains by failure reason
# ------------------------------------------------------------

rc_failure_captains_by_reason = (
    rc_failure_events
    .groupby("failure_reason")["captain_id"]
    .nunique()
    .sort_values(ascending=False)
    .rename("affected_captains")
    .reset_index()
)

rc_failure_captains_by_reason["pct_of_rc_failure_captains"] = (
    rc_failure_captains_by_reason["affected_captains"]
    / len(rc_failure_captains)
    * 100
)

rc_failure_captains_by_reason["pct_of_rc_failure_captains"] = (
    rc_failure_captains_by_reason[
        "pct_of_rc_failure_captains"
    ].round(2)
)

print("\nUnique captains affected by failure reason:")
display(rc_failure_captains_by_reason)

RC VERIFICATION FAILURE REASONS

Failure events by reason:


,failure_reason,failure_events,pct_of_failure_events
0,image_blurred,1028,28.980
1,ocr_low_confidence,797,22.470
2,name_mismatch,461,13.000
3,document_expired,366,10.320
4,details_not_legible,365,10.290
5,wrong_document_type,329,9.280
6,duplicate_document,201,5.670



Unique captains affected by failure reason:


,failure_reason,affected_captains,pct_of_rc_failure_captains
0,image_blurred,946,34.690
1,ocr_low_confidence,745,27.320
2,name_mismatch,446,16.350
3,document_expired,362,13.270
4,details_not_legible,353,12.940
5,wrong_document_type,323,11.840
6,duplicate_document,199,7.300


In [ ]:
# RC Dropout Segmentation

print("=" * 80)
print("RC DROPOUT SEGMENTATION")
print("=" * 80)


def segment_rc_dropout(df, column):
    """
    Compare RC dropout rate across segments.
    """

    result = (
        df.groupby(column)
        .agg(
            rc_entrants=("reached_DL", "size"),
            rc_completers=("reached_RC", "sum")
        )
        .reset_index()
    )

    result["rc_dropouts"] = (
        result["rc_entrants"]
        - result["rc_completers"]
    )

    result["rc_dropout_rate"] = (
        result["rc_dropouts"]
        / result["rc_entrants"]
        * 100
    )

    result["rc_completion_rate"] = (
        result["rc_completers"]
        / result["rc_entrants"]
        * 100
    )

    result["rc_dropout_rate"] = (
        result["rc_dropout_rate"].round(2)
    )

    result["rc_completion_rate"] = (
        result["rc_completion_rate"].round(2)
    )

    return result.sort_values(
        "rc_dropouts",
        ascending=False
    )


# Only captains who actually entered the RC stage
rc_analysis_base = funnel_base[
    funnel_base["reached_DL"]
].copy()


# ------------------------------------------------------------
# City
# ------------------------------------------------------------

print("\n1. BY CITY")
print("-" * 60)

city_rc = segment_rc_dropout(
    rc_analysis_base,
    "city"
)

display(city_rc)


# ------------------------------------------------------------
# Vehicle type
# ------------------------------------------------------------

print("\n2. BY VEHICLE TYPE")
print("-" * 60)

vehicle_rc = segment_rc_dropout(
    rc_analysis_base,
    "vehicle_type"
)

display(vehicle_rc)


# ------------------------------------------------------------
# Acquisition channel
# ------------------------------------------------------------

print("\n3. BY ACQUISITION CHANNEL")
print("-" * 60)

channel_rc = segment_rc_dropout(
    rc_analysis_base,
    "acquisition_channel"
)

display(channel_rc)


# ------------------------------------------------------------
# Device tier
# ------------------------------------------------------------

print("\n4. BY DEVICE TIER")
print("-" * 60)

device_rc = segment_rc_dropout(
    rc_analysis_base,
    "device_tier"
)

display(device_rc)


# ------------------------------------------------------------
# App language
# ------------------------------------------------------------

print("\n5. BY APP LANGUAGE")
print("-" * 60)

language_rc = segment_rc_dropout(
    rc_analysis_base,
    "app_language"
)

display(language_rc)


# ------------------------------------------------------------
# Age band
# ------------------------------------------------------------

print("\n6. BY AGE BAND")
print("-" * 60)

age_rc = segment_rc_dropout(
    rc_analysis_base,
    "age_band"
)

display(age_rc)

RC DROPOUT SEGMENTATION

1. BY CITY
------------------------------------------------------------


,city,rc_entrants,rc_completers,rc_dropouts,rc_dropout_rate,rc_completion_rate
2,Hyderabad,5470,3955,1515,27.700,72.300
1,Delhi,4618,3372,1246,26.980,73.020
0,Bangalore,3894,2809,1085,27.860,72.140
3,Pune,3913,2872,1041,26.600,73.400



2. BY VEHICLE TYPE
------------------------------------------------------------


,vehicle_type,rc_entrants,rc_completers,rc_dropouts,rc_dropout_rate,rc_completion_rate
0,Auto,8118,5907,2211,27.240,72.760
1,Cab,6365,4607,1758,27.620,72.380
2,ERickshaw,3412,2494,918,26.910,73.090



3. BY ACQUISITION CHANNEL
------------------------------------------------------------


...

,device_tier,rc_entrants,rc_completers,rc_dropouts,rc_dropout_rate,rc_completion_rate
1,low,8225,5532,2693,32.740,67.260
2,mid,6903,5272,1631,23.630,76.370
0,high,2767,2204,563,20.350,79.650



5. BY APP LANGUAGE
------------------------------------------------------------


,app_language,rc_entrants,rc_completers,rc_dropouts,rc_dropout_rate,rc_completion_rate
2,kn,3634,2628,1006,27.680,72.320
3,mr,3550,2551,999,28.140,71.860
4,te,3640,2668,972,26.700,73.300
0,en,3550,2587,963,27.130,72.870
1,hi,3521,2574,947,26.900,73.100



6. BY AGE BAND
------------------------------------------------------------


,age_band,rc_entrants,rc_completers,rc_dropouts,rc_dropout_rate,rc_completion_rate
1,25-34,7454,5415,2039,27.350,72.650
2,35-44,5047,3693,1354,26.830,73.170
0,18-24,3260,2339,921,28.250,71.750
3,45+,2134,1561,573,26.850,73.150


In [ ]:
# ============================================================
# LOW-DEVICE RC FAILURE REASONS
# ============================================================

# RC verification failures only
rc_failures = doc_events[
    (doc_events["doc_type"] == "RC") &
    (doc_events["event_type"] == "verification_fail")
].copy()

# Add captain attributes
rc_failures = rc_failures.merge(
    captains[
        ["captain_id", "device_tier", "city", "vehicle_type",
         "acquisition_channel", "app_language", "age_band"]
    ],
    on="captain_id",
    how="left"
)

# Failure events by device tier and reason
device_reason = (
    rc_failures
    .groupby(["device_tier", "failure_reason"])
    .size()
    .reset_index(name="failure_events")
    .sort_values(["device_tier", "failure_events"], ascending=[True, False])
)

print("RC VERIFICATION FAILURE EVENTS BY DEVICE TIER")
print("=" * 70)
display(device_reason)

# Top failure reasons within each device tier
print("\nTOP FAILURE REASONS WITHIN EACH DEVICE TIER")
print("=" * 70)

for device in ["low", "mid", "high"]:
    temp = (
        device_reason[device_reason["device_tier"] == device]
        .sort_values("failure_events", ascending=False)
        .head(5)
    )

    print(f"\n{device.upper()} DEVICE")
    display(temp)

RC VERIFICATION FAILURE EVENTS BY DEVICE TIER


,device_tier,failure_reason,failure_events
3,high,image_blurred,142
4,high,name_mismatch,134
1,high,document_expired,107
5,high,ocr_low_confidence,93
0,high,details_not_legible,71
6,high,wrong_document_type,71
2,high,duplicate_document,62
10,low,image_blurred,1709
12,low,ocr_low_confidence,1332
11,low,name_mismatch,504



TOP FAILURE REASONS WITHIN EACH DEVICE TIER

LOW DEVICE


,device_tier,failure_reason,failure_events
10,low,image_blurred,1709
12,low,ocr_low_confidence,1332
11,low,name_mismatch,504
7,low,details_not_legible,475
8,low,document_expired,406



MID DEVICE


,device_tier,failure_reason,failure_events
18,mid,name_mismatch,486
17,mid,image_blurred,437
15,mid,document_expired,397
19,mid,ocr_low_confidence,348
20,mid,wrong_document_type,320



HIGH DEVICE


,device_tier,failure_reason,failure_events
3,high,image_blurred,142
4,high,name_mismatch,134
1,high,document_expired,107
5,high,ocr_low_confidence,93
0,high,details_not_legible,71


In [ ]:
# ============================================================
# RC VERIFICATION FAILURE RATE BY DEVICE TIER
# ============================================================

# RC entrants = captains who passed DL

print("Available funnel columns:")
print(funnel_base.columns.tolist())

# Find the DL pass column
dl_candidates = [
    col for col in funnel_base.columns
    if "dl" in col.lower() and "pass" in col.lower()
]

print("\nPossible DL pass columns:", dl_candidates)

# Use the first matching DL pass column
dl_col = dl_candidates[0]

# RC entrants
rc_entrants_device = funnel_base[
    funnel_base[dl_col] == True
].copy()

# Captains who experienced at least one RC verification failure
rc_failure_captains = set(
    rc_failures["captain_id"].unique()
)

# Flag RC verification failure
rc_entrants_device["rc_verification_failed"] = (
    rc_entrants_device["captain_id"]
    .isin(rc_failure_captains)
)

# Summarize by device tier
device_failure_summary = (
    rc_entrants_device
    .groupby("device_tier")
    .agg(
        rc_entrants=("captain_id", "count"),
        captains_with_rc_failure=("rc_verification_failed", "sum")
    )
    .reset_index()
)

device_failure_summary["failure_rate"] = (
    device_failure_summary["captains_with_rc_failure"]
    / device_failure_summary["rc_entrants"]
    * 100
)

device_failure_summary["failure_rate"] = (
    device_failure_summary["failure_rate"].round(2)
)

device_failure_summary = device_failure_summary.sort_values(
    "failure_rate",
    ascending=False
)

print("\n" + "=" * 70)
print("RC VERIFICATION FAILURE RATE BY DEVICE TIER")
print("=" * 70)

display(device_failure_summary)

Available funnel columns:
['captain_id', 'signup_ts', 'city', 'vehicle_type', 'acquisition_channel', 'device_tier', 'app_language', 'age_band', 'DL_passed', 'RC_passed', 'AADHAAR_passed', 'PERMIT_passed', 'FITNESS_passed', 'INSURANCE_passed', 'furthest_stage_reached', 'reached_DL', 'reached_RC', 'reached_Aadhaar', 'reached_Permit', 'reached_Fitness', 'reached_Insurance']

Possible DL pass columns: ['DL_passed']

RC VERIFICATION FAILURE RATE BY DEVICE TIER


,device_tier,rc_entrants,captains_with_rc_failure,failure_rate
1,low,8225,3171,38.550
2,mid,6903,1773,25.680
0,high,2767,521,18.830


In [ ]:
# ============================================================
# LOW-DEVICE RC OPPORTUNITY SCENARIO
# ============================================================

# IMPORTANT:
# RC entrants = captains who passed DL.
# RC completers = captains who passed RC.
#
# We therefore use DL_passed as the RC-entry denominator.

rc_entrants_device = funnel_base[
    funnel_base["DL_passed"] == True
].copy()

# RC completion
rc_entrants_device["rc_completed"] = (
    rc_entrants_device["RC_passed"] == True
)

# ------------------------------------------------------------
# Calculate RC performance by device tier
# ------------------------------------------------------------

device_opportunity = (
    rc_entrants_device
    .groupby("device_tier")
    .agg(
        rc_entrants=("captain_id", "count"),
        rc_completers=("rc_completed", "sum")
    )
    .reset_index()
)

device_opportunity["rc_dropouts"] = (
    device_opportunity["rc_entrants"]
    - device_opportunity["rc_completers"]
)

device_opportunity["dropout_rate_pct"] = (
    device_opportunity["rc_dropouts"]
    / device_opportunity["rc_entrants"]
    * 100
)

device_opportunity = device_opportunity.sort_values(
    "dropout_rate_pct",
    ascending=False
)

print("=" * 70)
print("RC DROPOUT BY DEVICE TIER")
print("=" * 70)

display(device_opportunity)


# ------------------------------------------------------------
# LOW vs HIGH DEVICE BENCHMARK
# ------------------------------------------------------------

low = device_opportunity[
    device_opportunity["device_tier"] == "low"
].iloc[0]

high = device_opportunity[
    device_opportunity["device_tier"] == "high"
].iloc[0]

low_entrants = low["rc_entrants"]

low_dropout_rate = low["dropout_rate_pct"] / 100
high_dropout_rate = high["dropout_rate_pct"] / 100

dropout_gap = low_dropout_rate - high_dropout_rate

# Benchmark scenario:
# What if low-device RC dropout matched high-device RC dropout?

excess_rc_dropouts = (
    low_entrants * dropout_gap
)

# Mature cohort = January through May = 5 months
additional_rc_completions_monthly = (
    excess_rc_dropouts / 5
)

# Observed downstream conversion:
# RC entrants -> approved
rc_to_approval_rate = 3532 / 17895

additional_approvals_monthly = (
    additional_rc_completions_monthly
    * rc_to_approval_rate
)


# ------------------------------------------------------------
# DISPLAY SCENARIO
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("LOW-DEVICE RC OPPORTUNITY — BENCHMARK SCENARIO")
print("=" * 70)

print(f"Low-device RC entrants: "
      f"{low_entrants:,.0f}")

print(f"Low-device RC dropout rate: "
      f"{low_dropout_rate:.2%}")

print(f"High-device RC dropout rate: "
      f"{high_dropout_rate:.2%}")

print(f"Dropout gap: "
      f"{dropout_gap:.2%}")

print(f"\nEstimated excess RC dropouts in 5-month mature cohort: "
      f"{excess_rc_dropouts:,.0f}")

print(f"Estimated additional RC completions/month: "
      f"{additional_rc_completions_monthly:,.0f}")

print(f"Observed RC → approval conversion: "
      f"{rc_to_approval_rate:.2%}")

print(f"Benchmark additional approvals/month: "
      f"{additional_approvals_monthly:,.0f}")

RC DROPOUT BY DEVICE TIER


,device_tier,rc_entrants,rc_completers,rc_dropouts,dropout_rate_pct
1,low,8225,5532,2693,32.742
2,mid,6903,5272,1631,23.627
0,high,2767,2204,563,20.347



LOW-DEVICE RC OPPORTUNITY — BENCHMARK SCENARIO
Low-device RC entrants: 8,225
Low-device RC dropout rate: 32.74%
High-device RC dropout rate: 20.35%
Dropout gap: 12.39%

Estimated excess RC dropouts in 5-month mature cohort: 1,019
Estimated additional RC completions/month: 204
Observed RC → approval conversion: 19.74%
Benchmark additional approvals/month: 40


## A3 — Evaluate CAMP_WA_002

### Objective

Assess whether CAMP_WA_002 materially improves captain approval (A2O) and whether the evidence supports increasing campaign budget by 5×.

### Key questions

1. How does approval differ between campaign recipients and non-recipients?
2. Could the difference be explained by differences in onboarding stage, timing, or captain characteristics?
3. What is the most credible incremental estimate supported by the data?
4. Is the evidence strong enough to recommend a 5× budget increase?

### Principle

Separate observed association from incremental/causal impact. A higher approval rate among recipients does not by itself prove that the campaign caused the improvement.

In [ ]:
# ============================================================
# CAMP_WA_002 OVERVIEW
# ============================================================

campaign_id = "CAMP_WA_002"

# Campaign records
camp = nudges[
    nudges["campaign_id"] == campaign_id
].copy()

print("=" * 70)
print("CAMP_WA_002 — OVERVIEW")
print("=" * 70)

print(f"Total campaign records: {len(camp):,}")
print(f"Unique captains reached: {camp['captain_id'].nunique():,}")

print("\nChannel:")
display(
    camp["channel"]
    .value_counts()
    .rename_axis("channel")
    .reset_index(name="captains")
)

print("\nDelivery:")
display(
    camp["delivered"]
    .value_counts()
    .rename_axis("delivered")
    .reset_index(name="captains")
)

print("\nClick:")
display(
    camp["clicked"]
    .value_counts()
    .rename_axis("clicked")
    .reset_index(name="captains")
)

# Delivery and click rates
delivery_rate = camp["delivered"].mean() * 100
click_rate_all = camp["clicked"].mean() * 100

# Click-through among delivered messages
delivered_camp = camp[camp["delivered"] == True]

if len(delivered_camp) > 0:
    click_rate_delivered = (
        delivered_camp["clicked"].mean() * 100
    )
else:
    click_rate_delivered = 0

print("\n" + "=" * 70)
print("CAMPAIGN METRICS")
print("=" * 70)

print(f"Delivery rate: {delivery_rate:.2f}%")
print(f"Click rate (all sends): {click_rate_all:.2f}%")
print(f"Click rate (delivered): {click_rate_delivered:.2f}%")

CAMP_WA_002 — OVERVIEW
Total campaign records: 8,673
Unique captains reached: 8,673

Channel:


,channel,captains
0,whatsapp,8673



Delivery:


,delivered,captains
0,1,8058
1,0,615



Click:


,clicked,captains
0,0,5104
1,1,3569



CAMPAIGN METRICS
Delivery rate: 92.91%
Click rate (all sends): 41.15%
Click rate (delivered): 41.18%


In [ ]:
# ============================================================
# CAMP_WA_002: RAW APPROVAL COMPARISON
# ============================================================

campaign_id = "CAMP_WA_002"

# ------------------------------------------------------------
# Define mature cohort explicitly
# Extraction cutoff = 30-Jun-2026 23:59
# Mature cohort = signups through 31-May-2026 23:59
# ------------------------------------------------------------

maturity_cutoff = pd.Timestamp("2026-05-31 23:59:00")

mature_captains = captains[
    captains["signup_ts"] <= maturity_cutoff
].copy()

print(f"Mature cohort size: {len(mature_captains):,}")


# ------------------------------------------------------------
# Campaign exposure
# ------------------------------------------------------------

campaign_recipients = set(
    nudges.loc[
        nudges["campaign_id"] == campaign_id,
        "captain_id"
    ]
)

mature_captains["camp_wa_002"] = (
    mature_captains["captain_id"].isin(campaign_recipients)
)


# ------------------------------------------------------------
# Approval outcome
# ------------------------------------------------------------

mature_captains = mature_captains.merge(
    approvals[["captain_id", "final_status"]],
    on="captain_id",
    how="left"
)

mature_captains["approved"] = (
    mature_captains["final_status"] == "approved"
)


# ------------------------------------------------------------
# Raw recipient vs non-recipient comparison
# ------------------------------------------------------------

campaign_summary = (
    mature_captains
    .groupby("camp_wa_002")
    .agg(
        captains=("captain_id", "count"),
        approved=("approved", "sum")
    )
    .reset_index()
)

campaign_summary["approval_rate_pct"] = (
    campaign_summary["approved"]
    / campaign_summary["captains"]
    * 100
)

campaign_summary["group"] = campaign_summary["camp_wa_002"].map({
    True: "CAMP_WA_002 recipient",
    False: "Non-recipient"
})

campaign_summary = campaign_summary[
    ["group", "captains", "approved", "approval_rate_pct"]
]

print("\n" + "=" * 70)
print("CAMP_WA_002 — RAW APPROVAL COMPARISON")
print("=" * 70)

display(campaign_summary)


# ------------------------------------------------------------
# Raw difference
# ------------------------------------------------------------

recipient_rate = campaign_summary.loc[
    campaign_summary["group"] == "CAMP_WA_002 recipient",
    "approval_rate_pct"
].iloc[0]

nonrecipient_rate = campaign_summary.loc[
    campaign_summary["group"] == "Non-recipient",
    "approval_rate_pct"
].iloc[0]

raw_difference = recipient_rate - nonrecipient_rate

print("\n" + "=" * 70)
print("RAW APPROVAL DIFFERENCE")
print("=" * 70)

print(f"Recipient approval rate: {recipient_rate:.2f}%")
print(f"Non-recipient approval rate: {nonrecipient_rate:.2f}%")
print(f"Raw difference: +{raw_difference:.2f} percentage points")

Mature cohort size: 20,207

CAMP_WA_002 — RAW APPROVAL COMPARISON


,group,captains,approved,approval_rate_pct
0,Non-recipient,13079,1466,11.209
1,CAMP_WA_002 recipient,7128,2066,28.984



RAW APPROVAL DIFFERENCE
Recipient approval rate: 28.98%
Non-recipient approval rate: 11.21%
Raw difference: +17.78 percentage points


In [ ]:
# ============================================================
# CAMP_WA_002 TIMING & ONBOARDING STAGE
# ============================================================

campaign_id = "CAMP_WA_002"

# Campaign records
camp = nudges[
    nudges["campaign_id"] == campaign_id
].copy()

# Add signup timestamp
camp = camp.merge(
    captains[["captain_id", "signup_ts"]],
    on="captain_id",
    how="left"
)

# Time from signup to campaign send
camp["hours_after_signup"] = (
    camp["sent_ts"] - camp["signup_ts"]
).dt.total_seconds() / 3600

# ------------------------------------------------------------
# Campaign timing
# ------------------------------------------------------------

print("=" * 70)
print("CAMP_WA_002 — TIMING")
print("=" * 70)

print(f"Median hours after signup: "
      f"{camp['hours_after_signup'].median():.2f}")

print(f"Mean hours after signup: "
      f"{camp['hours_after_signup'].mean():.2f}")

print(f"Minimum hours after signup: "
      f"{camp['hours_after_signup'].min():.2f}")

print(f"Maximum hours after signup: "
      f"{camp['hours_after_signup'].max():.2f}")


# ------------------------------------------------------------
# Determine document stage immediately before campaign send
# ------------------------------------------------------------

# RC-relevant document sequence.
# We count successful verification passes BEFORE campaign send.

doc_types = [
    "DL",
    "RC",
    "AADHAAR",
    "PERMIT",
    "FITNESS",
    "INSURANCE"
]

# Merge campaign sends with document events
camp_docs = camp[
    ["captain_id", "sent_ts"]
].merge(
    doc_events[
        [
            "captain_id",
            "doc_type",
            "event_type",
            "event_ts"
        ]
    ],
    on="captain_id",
    how="left"
)

# Only successful verification passes before campaign send
camp_docs = camp_docs[
    (camp_docs["event_type"] == "verification_pass") &
    (camp_docs["event_ts"] < camp_docs["sent_ts"]) &
    (camp_docs["doc_type"].isin(doc_types))
].copy()

# Number of distinct documents passed before campaign send
docs_passed = (
    camp_docs
    .groupby("captain_id")["doc_type"]
    .nunique()
    .rename("docs_passed_before_campaign")
)

camp = camp.merge(
    docs_passed,
    on="captain_id",
    how="left"
)

camp["docs_passed_before_campaign"] = (
    camp["docs_passed_before_campaign"].fillna(0)
)

# ------------------------------------------------------------
# Distribution of onboarding stage at campaign send
# ------------------------------------------------------------

stage_distribution = (
    camp["docs_passed_before_campaign"]
    .value_counts()
    .sort_index()
    .reset_index()
)

stage_distribution.columns = [
    "docs_passed_before_campaign",
    "captains"
]

stage_distribution["pct"] = (
    stage_distribution["captains"]
    / len(camp)
    * 100
)

print("\n" + "=" * 70)
print("ONBOARDING STAGE AT CAMP_WA_002 SEND")
print("=" * 70)

display(stage_distribution)

CAMP_WA_002 — TIMING
Median hours after signup: 53.53
Mean hours after signup: 56.72
Minimum hours after signup: 9.40
Maximum hours after signup: 184.94

ONBOARDING STAGE AT CAMP_WA_002 SEND


,docs_passed_before_campaign,captains,pct
0,2,7734,89.173
1,3,892,10.285
2,4,44,0.507
3,5,3,0.035


In [ ]:
# ============================================================
# MATURE 2-DOC CAMPAIGN COMPARISON
# ============================================================

campaign_id = "CAMP_WA_002"

# ------------------------------------------------------------
# 1. Mature cohort
# ------------------------------------------------------------

maturity_cutoff = pd.Timestamp("2026-05-31 23:59:00")

analysis_captains = captains[
    captains["signup_ts"] <= maturity_cutoff
].copy()

# Approval outcome
analysis_captains = analysis_captains.merge(
    approvals[["captain_id", "final_status"]],
    on="captain_id",
    how="left"
)

analysis_captains["approved"] = (
    analysis_captains["final_status"] == "approved"
)


# ------------------------------------------------------------
# 2. CAMP_WA_002 send information
# ------------------------------------------------------------

camp = nudges[
    nudges["campaign_id"] == campaign_id
].copy()

camp = camp.merge(
    captains[["captain_id", "signup_ts"]],
    on="captain_id",
    how="left"
)

camp["hours_after_signup"] = (
    camp["sent_ts"] - camp["signup_ts"]
).dt.total_seconds() / 3600


# ------------------------------------------------------------
# 3. Calculate documents passed before campaign send
# ------------------------------------------------------------

camp_docs = camp[
    ["captain_id", "sent_ts"]
].merge(
    doc_events[
        [
            "captain_id",
            "doc_type",
            "event_type",
            "event_ts"
        ]
    ],
    on="captain_id",
    how="left"
)

camp_docs = camp_docs[
    (camp_docs["event_type"] == "verification_pass") &
    (camp_docs["event_ts"] < camp_docs["sent_ts"])
].copy()

docs_passed = (
    camp_docs
    .groupby("captain_id")["doc_type"]
    .nunique()
    .rename("docs_passed_before_campaign")
)

camp = camp.merge(
    docs_passed,
    on="captain_id",
    how="left"
)

camp["docs_passed_before_campaign"] = (
    camp["docs_passed_before_campaign"]
    .fillna(0)
)


# ------------------------------------------------------------
# 4. Keep only mature CAMP_WA_002 recipients at 2-doc stage
# ------------------------------------------------------------

recipient_2doc = camp.merge(
    analysis_captains[
        [
            "captain_id",
            "city",
            "vehicle_type",
            "acquisition_channel",
            "device_tier",
            "app_language",
            "age_band",
            "approved"
        ]
    ],
    on="captain_id",
    how="inner"
)

recipient_2doc = recipient_2doc[
    recipient_2doc["docs_passed_before_campaign"] == 2
].copy()

print("=" * 70)
print("MATURE CAMP_WA_002 RECIPIENTS AT 2-DOC STAGE")
print("=" * 70)

print(f"Recipients at 2-doc stage: {len(recipient_2doc):,}")
print(f"Approval rate: {recipient_2doc['approved'].mean() * 100:.2f}%")

MATURE CAMP_WA_002 RECIPIENTS AT 2-DOC STAGE
Recipients at 2-doc stage: 6,362
Approval rate: 28.67%


In [ ]:
# ============================================================
# PSEUDO-CONTROL GROUP AT 2-DOC STAGE
# ============================================================

campaign_id = "CAMP_WA_002"

# Use the observed median campaign delay as the pseudo-treatment time
campaign_delay_hours = 54


# ------------------------------------------------------------
# 1. Mature cohort
# ------------------------------------------------------------

maturity_cutoff = pd.Timestamp("2026-05-31 23:59:00")

control = captains[
    captains["signup_ts"] <= maturity_cutoff
].copy()


# ------------------------------------------------------------
# 2. Exclude ALL CAMP_WA_002 recipients
# ------------------------------------------------------------

all_campaign_recipients = set(
    nudges.loc[
        nudges["campaign_id"] == campaign_id,
        "captain_id"
    ]
)

control = control[
    ~control["captain_id"].isin(all_campaign_recipients)
].copy()


# ------------------------------------------------------------
# 3. Create pseudo-campaign timestamp
# ------------------------------------------------------------

control["pseudo_campaign_ts"] = (
    control["signup_ts"]
    + pd.to_timedelta(
        campaign_delay_hours,
        unit="h"
    )
)


# ------------------------------------------------------------
# 4. Find documents passed before pseudo-campaign time
# ------------------------------------------------------------

control_docs = control[
    ["captain_id", "pseudo_campaign_ts"]
].merge(
    doc_events[
        [
            "captain_id",
            "doc_type",
            "event_type",
            "event_ts"
        ]
    ],
    on="captain_id",
    how="left"
)

control_docs = control_docs[
    (control_docs["event_type"] == "verification_pass") &
    (control_docs["event_ts"] < control_docs["pseudo_campaign_ts"])
].copy()


# Count distinct documents successfully passed
control_docs_count = (
    control_docs
    .groupby("captain_id")["doc_type"]
    .nunique()
    .rename("docs_passed_before_pseudo_campaign")
)

control = control.merge(
    control_docs_count,
    on="captain_id",
    how="left"
)

control["docs_passed_before_pseudo_campaign"] = (
    control["docs_passed_before_pseudo_campaign"]
    .fillna(0)
)


# ------------------------------------------------------------
# 5. Keep captains at exactly 2-doc stage
# ------------------------------------------------------------

control_2doc = control[
    control["docs_passed_before_pseudo_campaign"] == 2
].copy()


# ------------------------------------------------------------
# 6. Add approval outcome
# ------------------------------------------------------------

control_2doc = control_2doc.merge(
    approvals[
        ["captain_id", "decision_ts", "final_status"]
    ],
    on="captain_id",
    how="left"
)

control_2doc["approved"] = (
    control_2doc["final_status"] == "approved"
)


# ------------------------------------------------------------
# 7. Exclude captains already approved before pseudo-send
# ------------------------------------------------------------

control_2doc["approved_before_pseudo"] = (
    (control_2doc["final_status"] == "approved") &
    (control_2doc["decision_ts"].notna()) &
    (control_2doc["decision_ts"] <= control_2doc["pseudo_campaign_ts"])
)

control_2doc = control_2doc[
    ~control_2doc["approved_before_pseudo"]
].copy()


# ------------------------------------------------------------
# 8. Result
# ------------------------------------------------------------

print("=" * 70)
print("PSEUDO-CONTROL — MATURE NON-RECIPIENTS AT 2-DOC STAGE")
print("=" * 70)

print(f"Non-recipient 2-doc controls: "
      f"{len(control_2doc):,}")

print(f"Final approval rate: "
      f"{control_2doc['approved'].mean() * 100:.2f}%")

PSEUDO-CONTROL — MATURE NON-RECIPIENTS AT 2-DOC STAGE
Non-recipient 2-doc controls: 2,573
Final approval rate: 22.43%


In [ ]:
# A3.5 — Treatment vs Control Segment Composition

print("=" * 70)
print("TREATMENT vs CONTROL — SEGMENT COMPOSITION")
print("=" * 70)

print(f"\nTreatment size: {len(recipient_2doc):,}")
print(f"Control size:   {len(control_2doc):,}")

segment_columns = [
    "city",
    "vehicle_type",
    "acquisition_channel",
    "device_tier",
    "app_language",
    "age_band"
]

for col in segment_columns:

    treatment_dist = (
        recipient_2doc[col]
        .value_counts(normalize=True)
        .mul(100)
        .rename("treatment_pct")
    )

    control_dist = (
        mature_captains[
            mature_captains["captain_id"].isin(
                control_2doc["captain_id"]
            )
        ][col]
        .value_counts(normalize=True)
        .mul(100)
        .rename("control_pct")
    )

    comparison = pd.concat(
        [treatment_dist, control_dist],
        axis=1
    ).fillna(0)

    comparison["abs_difference_pp"] = (
        comparison["treatment_pct"]
        - comparison["control_pct"]
    ).abs()

    print(f"\n{col}")
    display(comparison.round(2))

    print(
        f"Maximum absolute difference: "
        f"{comparison['abs_difference_pp'].max():.2f} pp"
    )

TREATMENT vs CONTROL — SEGMENT COMPOSITION

Treatment size: 6,362
Control size:   2,573

city


,treatment_pct,control_pct,abs_difference_pp
city,,,
Hyderabad,30.750,30.430,0.310
Delhi,25.450,26.390,0.940
Pune,22.460,22.040,0.420
Bangalore,21.350,21.140,0.200


Maximum absolute difference: 0.94 pp

vehicle_type


,treatment_pct,control_pct,abs_difference_pp
vehicle_type,,,
Auto,45.600,46.440,0.840
Cab,35.740,34.820,0.920
ERickshaw,18.660,18.730,0.080


Maximum absolute difference: 0.92 pp

acquisition_channel


...

Maximum absolute difference: 1.85 pp

app_language


,treatment_pct,control_pct,abs_difference_pp
app_language,,,
te,20.940,19.940,1.000
kn,20.560,20.440,0.120
mr,19.710,19.550,0.160
en,19.490,20.130,0.640
hi,19.300,19.940,0.640


Maximum absolute difference: 1.00 pp

age_band


,treatment_pct,control_pct,abs_difference_pp
age_band,,,
25-34,42.080,40.650,1.430
35-44,29.130,27.560,1.570
18-24,17.530,18.540,1.010
45+,11.270,13.250,1.980


Maximum absolute difference: 1.98 pp


## A4 — Ranked Recommendations

Based on the funnel, RC-stage diagnostics, device-tier segmentation, and campaign evaluation, prioritize interventions by expected impact, actionability, and evidence strength.

For each recommendation:
- Action
- Expected impact and calculation
- Cost / risk
- Success metric

In [89]:
# ============================================================
# A4 RECOMMENDATION IMPACT CALCULATIONS
# ============================================================

# ------------------------------------------------------------
# RECOMMENDATION 1 — Low-device RC onboarding
# ------------------------------------------------------------

# Values already calculated in the device-tier analysis
low_device_rc_entrants = low["rc_entrants"]
low_device_dropout_rate = low["dropout_rate_pct"] / 100
high_device_dropout_rate = high["dropout_rate_pct"] / 100

rc_dropout_gap = (
    low_device_dropout_rate -
    high_device_dropout_rate
)

benchmark_recovered_rc = (
    low_device_rc_entrants *
    rc_dropout_gap
)

# Mature cohort covers 5 months
monthly_rc_recovery = (
    benchmark_recovered_rc / 5
)

# Observed RC → approval conversion
rc_to_approval = (
    approved_count / rc_count
)

monthly_approval_opportunity = (
    monthly_rc_recovery *
    rc_to_approval
)


# ------------------------------------------------------------
# RECOMMENDATION 2 — RC verification failure reduction
# ------------------------------------------------------------

# RC verification failure events already calculated earlier
total_rc_failure_events = len(rc_failure_events)

image_blurred_events = (
    rc_failure_events["failure_reason"]
    .eq("image_blurred")
    .sum()
)

ocr_events = (
    rc_failure_events["failure_reason"]
    .eq("ocr_low_confidence")
    .sum()
)

image_ocr_share = (
    (image_blurred_events + ocr_events)
    / total_rc_failure_events
)

# RC dropouts already calculated earlier
total_rc_dropouts = len(rc_dropouts)

failure_captain_ids = set(
    rc_failure_events["captain_id"].unique()
)

rc_dropouts_with_failure_count = (
    rc_dropouts["captain_id"]
    .isin(failure_captain_ids)
    .sum()
)

failure_associated_dropout_share = (
    rc_dropouts_with_failure_count /
    total_rc_dropouts
)


# ------------------------------------------------------------
# RECOMMENDATION 3 — CAMP_WA_002 experiment
# ------------------------------------------------------------

# Use validated results from the campaign analysis
raw_campaign_difference = (
    recipient_rate -
    nonrecipient_rate
)

stage_controlled_difference = (
    recipient_2doc["approved"].mean() * 100 -
    control_2doc["approved"].mean() * 100
)


# ------------------------------------------------------------
# DISPLAY
# ------------------------------------------------------------

print("=" * 70)
print("A4 — RECOMMENDATION IMPACT CALCULATIONS")
print("=" * 70)

print("\n1. LOW-DEVICE RC BENCHMARK")
print("-" * 70)

print(
    f"Low-device RC entrants: "
    f"{low_device_rc_entrants:,.0f}"
)

print(
    f"Low-device RC dropout rate: "
    f"{low_device_dropout_rate:.2%}"
)

print(
    f"High-device RC dropout rate: "
    f"{high_device_dropout_rate:.2%}"
)

print(
    f"Dropout gap: "
    f"{rc_dropout_gap:.2%}"
)

print(
    f"Benchmark RC completions recovered / 5 months: "
    f"{benchmark_recovered_rc:,.0f}"
)

print(
    f"Benchmark RC completions recovered / month: "
    f"{monthly_rc_recovery:,.0f}"
)

print(
    f"Observed RC → approval conversion: "
    f"{rc_to_approval:.2%}"
)

print(
    f"Benchmark additional approvals / month: "
    f"{monthly_approval_opportunity:,.0f}"
)


print("\n2. RC VERIFICATION FAILURE OPPORTUNITY")
print("-" * 70)

print(
    f"Total RC verification failure events: "
    f"{total_rc_failure_events:,}"
)

print(
    f"Image-blurred + OCR failure-event share: "
    f"{image_ocr_share:.2%}"
)

print(
    f"RC dropouts with recorded verification failure: "
    f"{failure_associated_dropout_share:.2%}"
)


print("\n3. CAMP_WA_002")
print("-" * 70)

print(
    f"Raw approval difference: "
    f"+{raw_campaign_difference:.2f} pp"
)

print(
    f"Stage/timing-controlled difference: "
    f"+{stage_controlled_difference:.2f} pp"
)

A4 — RECOMMENDATION IMPACT CALCULATIONS

1. LOW-DEVICE RC BENCHMARK
----------------------------------------------------------------------
Low-device RC entrants: 8,225
Low-device RC dropout rate: 32.74%
High-device RC dropout rate: 20.35%
Dropout gap: 12.39%
Benchmark RC completions recovered / 5 months: 1,019
Benchmark RC completions recovered / month: 204
Observed RC → approval conversion: 27.15%
Benchmark additional approvals / month: 55

2. RC VERIFICATION FAILURE OPPORTUNITY
----------------------------------------------------------------------
Total RC verification failure events: 8,174
Image-blurred + OCR failure-event share: 49.68%
RC dropouts with recorded verification failure: 55.80%

3. CAMP_WA_002
----------------------------------------------------------------------
Raw approval difference: +17.78 pp
Stage/timing-controlled difference: +6.25 pp


## Part B — Optional Associate-Level Supply Analysis

### B1 — Airport Demand–Supply Mismatch

Objective:
Characterize when airport demand exceeds available captain supply, quantify the
size of the mismatch, and identify the time windows where supply intervention
could have the highest potential impact.

Important limitation:
`airport_hourly.csv` provides aggregate hourly supply and demand, not individual
captain-level availability. Any implied incremental supply requirement is therefore
a scenario estimate, not a causal staffing requirement.

In [ ]:
# B1.1 — Overall airport demand-supply baseline

import pandas as pd

# Load raw airport hourly data
airport = airport_hourly

# Parse timestamp
airport["hour_ts"] = pd.to_datetime(airport["hour_ts"], errors="coerce")

# Basic validation
print("Dataset shape:", airport.shape)
print("Missing values:", airport.isna().sum().sum())
print("Time range:", airport["hour_ts"].min(), "to", airport["hour_ts"].max())

# Overall demand-supply totals
total_requests = airport["requests"].sum()
total_fulfilled = airport["fulfilled_requests"].sum()
total_unfulfilled = airport["unfulfilled_requests"].sum()

fulfillment_rate = total_fulfilled / total_requests * 100
unfulfilled_rate = total_unfulfilled / total_requests * 100

print("\nAirport Demand-Supply Baseline")
print("-" * 40)
print(f"Total requests:           {total_requests:,.0f}")
print(f"Fulfilled requests:       {total_fulfilled:,.0f}")
print(f"Unfulfilled requests:     {total_unfulfilled:,.0f}")
print(f"Fulfillment rate:         {fulfillment_rate:.1f}%")
print(f"Unfulfilled rate:          {unfulfilled_rate:.1f}%")
print(f"Airport hourly records:   {len(airport):,}")
print(f"Terminal zones:           {airport['zone_id'].nunique()}")

Dataset shape: (10248, 9)
Missing values: 0
Time range: 2026-05-01 00:00:00 to 2026-06-30 23:00:00

Airport Demand-Supply Baseline
----------------------------------------
Total requests:           457,610
Fulfilled requests:       393,269
Unfulfilled requests:     64,341
Fulfillment rate:         85.9%
Unfulfilled rate:          14.1%
Airport hourly records:   10,248
Terminal zones:           7


In [ ]:
# B1.2 — Airport demand-supply mismatch by hour

hourly = (
    airport
    .assign(hour=airport["hour_ts"].dt.hour)
    .groupby("hour", as_index=False)
    .agg(
        requests=("requests", "sum"),
        fulfilled_requests=("fulfilled_requests", "sum"),
        unfulfilled_requests=("unfulfilled_requests", "sum"),
        online_captains=("online_captains", "sum"),
        avg_eta_min=("avg_eta_min", "mean"),
        avg_surge_multiplier=("avg_surge_multiplier", "mean")
    )
)

hourly["fulfillment_rate"] = (
    hourly["fulfilled_requests"] / hourly["requests"] * 100
)

hourly["unfulfilled_rate"] = (
    hourly["unfulfilled_requests"] / hourly["requests"] * 100
)

hourly["request_per_online_captain"] = (
    hourly["requests"] / hourly["online_captains"]
)

display(
    hourly[
        [
            "hour",
            "requests",
            "fulfilled_requests",
            "unfulfilled_requests",
            "fulfillment_rate",
            "online_captains",
            "avg_eta_min",
            "avg_surge_multiplier"
        ]
    ].round(2)
)

,hour,requests,fulfilled_requests,unfulfilled_requests,fulfillment_rate,online_captains,avg_eta_min,avg_surge_multiplier
0,0,19378,12844,6534,66.280,10907,5.320,1.340
1,1,21560,13040,8520,60.480,10829,5.440,1.360
2,2,19703,12929,6774,65.620,10980,5.360,1.330
3,3,16363,12739,3624,77.850,10949,4.840,1.260
4,4,15314,13426,1888,87.670,11448,4.420,1.170
5,5,16466,14676,1790,89.130,12317,4.280,1.140
6,6,19203,16926,2277,88.140,13765,4.290,1.150
7,7,21850,20305,1545,92.930,16232,3.950,1.100
8,8,22887,22150,737,96.780,18769,3.690,1.050
9,9,21848,21303,545,97.510,19903,3.740,1.040


In [ ]:
# B1.3 — Quantify concentrated shortage windows

shortage_hours = [21, 22, 23, 0, 1, 2]

shortage = hourly[hourly["hour"].isin(shortage_hours)].copy()

shortage_unfulfilled = shortage["unfulfilled_requests"].sum()
total_unfulfilled = hourly["unfulfilled_requests"].sum()

shortage_share = shortage_unfulfilled / total_unfulfilled * 100

shortage_requests = shortage["requests"].sum()
shortage_fulfilled = shortage["fulfilled_requests"].sum()

shortage_fulfillment_rate = (
    shortage_fulfilled / shortage_requests * 100
)

hours_share = len(shortage_hours) / 24 * 100

print("Airport Shortage Window: 21:00–02:00")
print("-" * 45)
print(f"Requests:                    {shortage_requests:,.0f}")
print(f"Fulfilled requests:          {shortage_fulfilled:,.0f}")
print(f"Unfulfilled requests:        {shortage_unfulfilled:,.0f}")
print(f"Fulfillment rate:            {shortage_fulfillment_rate:.1f}%")
print(f"Share of all unfulfilled:    {shortage_share:.1f}%")
print(f"Share of daily hours:        {hours_share:.1f}%")

Airport Shortage Window: 21:00–02:00
---------------------------------------------
Requests:                    136,478
Fulfilled requests:          91,387
Unfulfilled requests:        45,091
Fulfillment rate:            67.0%
Share of all unfulfilled:    70.1%
Share of daily hours:        25.0%


In [ ]:
# B1.4 — Identify shortage concentration by airport zone

zone_hourly = (
    airport
    .assign(hour=airport["hour_ts"].dt.hour)
    .groupby(["zone_id", "hour"], as_index=False)
    .agg(
        requests=("requests", "sum"),
        fulfilled_requests=("fulfilled_requests", "sum"),
        unfulfilled_requests=("unfulfilled_requests", "sum"),
        online_captains=("online_captains", "sum"),
        avg_eta_min=("avg_eta_min", "mean"),
        avg_surge_multiplier=("avg_surge_multiplier", "mean")
    )
)

zone_hourly["fulfillment_rate"] = (
    zone_hourly["fulfilled_requests"]
    / zone_hourly["requests"]
    * 100
)

shortage_zone = zone_hourly[
    zone_hourly["hour"].isin([21, 22, 23, 0, 1, 2])
].copy()

zone_summary = (
    shortage_zone
    .groupby("zone_id", as_index=False)
    .agg(
        requests=("requests", "sum"),
        fulfilled_requests=("fulfilled_requests", "sum"),
        unfulfilled_requests=("unfulfilled_requests", "sum"),
        online_captains=("online_captains", "sum"),
        avg_eta_min=("avg_eta_min", "mean"),
        avg_surge_multiplier=("avg_surge_multiplier", "mean")
    )
)

zone_summary["fulfillment_rate"] = (
    zone_summary["fulfilled_requests"]
    / zone_summary["requests"]
    * 100
)

zone_summary["share_of_shortage_unfulfilled"] = (
    zone_summary["unfulfilled_requests"]
    / zone_summary["unfulfilled_requests"].sum()
    * 100
)

display(
    zone_summary.sort_values(
        "unfulfilled_requests",
        ascending=False
    ).round(2)
)

,zone_id,requests,fulfilled_requests,unfulfilled_requests,online_captains,avg_eta_min,avg_surge_multiplier,fulfillment_rate,share_of_shortage_unfulfilled
1,APT-T2,30787,9324,21463,4751,9.630,2.090,30.290,47.600
0,APT-T1,30613,9330,21283,4715,9.590,2.080,30.480,47.200
3,CBD-02,21323,20767,556,18141,3.620,1.040,97.390,1.230
2,CBD-01,21156,20630,526,17923,3.690,1.040,97.510,1.170
5,SUB-11,11112,10661,451,9991,3.650,1.050,95.940,1.000
4,SUB-07,11477,11036,441,10110,3.790,1.050,96.160,0.980
6,TECH-03,10010,9639,371,11352,3.680,1.040,96.290,0.820


In [ ]:
# B1.5 — Airport terminal shortage by hour

airport_terminal = zone_hourly[
    zone_hourly["zone_id"].isin(["APT-T1", "APT-T2"])
].copy()

airport_terminal_summary = (
    airport_terminal
    .groupby("hour", as_index=False)
    .agg(
        requests=("requests", "sum"),
        fulfilled_requests=("fulfilled_requests", "sum"),
        unfulfilled_requests=("unfulfilled_requests", "sum"),
        online_captains=("online_captains", "sum"),
        avg_eta_min=("avg_eta_min", "mean"),
        avg_surge_multiplier=("avg_surge_multiplier", "mean")
    )
)

airport_terminal_summary["fulfillment_rate"] = (
    airport_terminal_summary["fulfilled_requests"]
    / airport_terminal_summary["requests"]
    * 100
)

airport_terminal_summary["unfulfilled_share"] = (
    airport_terminal_summary["unfulfilled_requests"]
    / airport_terminal_summary["unfulfilled_requests"].sum()
    * 100
)

display(
    airport_terminal_summary[
        [
            "hour",
            "requests",
            "fulfilled_requests",
            "unfulfilled_requests",
            "fulfillment_rate",
            "online_captains",
            "avg_eta_min",
            "avg_surge_multiplier",
            "unfulfilled_share"
        ]
    ].round(2)
)

,hour,requests,fulfilled_requests,unfulfilled_requests,fulfillment_rate,online_captains,avg_eta_min,avg_surge_multiplier,unfulfilled_share
0,0,9124,2933,6191,32.150,1537,9.540,2.070,11.240
1,1,11288,3143,8145,27.840,1567,9.830,2.150,14.790
2,2,9500,3073,6427,32.350,1612,9.580,2.060,11.670
3,3,6190,2916,3274,47.110,1613,8.010,1.800,5.950
4,4,4718,3219,1499,68.230,1828,6.180,1.470,2.720
5,5,5291,3839,1452,72.560,2145,5.830,1.410,2.640
6,6,6776,4832,1944,71.310,2665,5.880,1.430,3.530
7,7,7459,6270,1189,84.060,3626,4.760,1.230,2.160
8,8,6771,6406,365,94.610,4484,3.920,1.070,0.660
9,9,5147,5025,122,97.630,5621,3.780,1.040,0.220


## B2 — Post-Airport Trip Behavior

Objective:
Assess whether airport trip destinations show different post-trip outcomes that
could help explain captain supply availability.

Because `airport_trips.csv` does not contain `captain_id`, this is a trip-level
analysis rather than individual captain longitudinal behavior. Return-fare
availability is treated as a proxy for the attractiveness of remaining in the
airport supply ecosystem after completing a trip.

In [ ]:
# B2.1 — Inspect airport trips schema

trips = airport_trips

print("Shape:", trips.shape)
print("\nColumns:")
print(trips.columns.tolist())

print("\nFirst 5 rows:")
display(trips.head())

Shape: (60000, 9)

Columns:
['trip_id', 'pickup_zone_id', 'drop_zone_id', 'drop_zone_type', 'request_ts', 'trip_distance_km', 'captain_cancelled', 'got_return_fare_within_20min', 'fare_inr']

First 5 rows:


,trip_id,pickup_zone_id,drop_zone_id,drop_zone_type,request_ts,trip_distance_km,captain_cancelled,got_return_fare_within_20min,fare_inr
0,TRP00036169,APT-T1,CBD-02,city_core,2026-05-01,8.180,0,1,148.000
1,TRP00018764,APT-T1,SUB-07,suburban,2026-05-01,21.100,0,0,302.000
2,TRP00027350,APT-T2,TECH-03,tech_park,2026-05-01,14.220,0,0,231.000
3,TRP00058214,APT-T2,TECH-03,tech_park,2026-05-01,9.810,0,0,207.000
4,TRP00044709,APT-T1,SUB-07,suburban,2026-05-01,18.020,1,0,333.000


In [ ]:
# B2.1 — Overall airport trip baseline

trips = airport_trips

trips["request_ts"] = pd.to_datetime(
    trips["request_ts"],
    errors="coerce"
)

total_trips = len(trips)

cancellation_rate = (
    trips["captain_cancelled"].mean() * 100
)

return_fare_rate = (
    trips["got_return_fare_within_20min"].mean() * 100
)

print("Airport Trip Baseline")
print("-" * 40)
print(f"Total airport trips:              {total_trips:,}")
print(f"Captain cancellation rate:        {cancellation_rate:.1f}%")
print(f"Return fare within 20 min:        {return_fare_rate:.1f}%")
print(
    f"Trips without return fare:        "
    f"{(1 - trips['got_return_fare_within_20min'].mean()) * 100:.1f}%"
)

Airport Trip Baseline
----------------------------------------
Total airport trips:              60,000
Captain cancellation rate:        13.8%
Return fare within 20 min:        36.0%
Trips without return fare:        64.0%


In [ ]:
# B2.2 — Post-airport outcomes by destination type

destination_summary = (
    trips
    .groupby("drop_zone_type", as_index=False)
    .agg(
        trips=("trip_id", "count"),
        cancellations=("captain_cancelled", "sum"),
        cancellation_rate=("captain_cancelled", "mean"),
        return_fares=("got_return_fare_within_20min", "sum"),
        return_fare_rate=("got_return_fare_within_20min", "mean"),
        avg_distance_km=("trip_distance_km", "mean"),
        avg_fare_inr=("fare_inr", "mean")
    )
)

destination_summary["cancellation_rate"] *= 100
destination_summary["return_fare_rate"] *= 100

destination_summary["share_of_trips"] = (
    destination_summary["trips"]
    / destination_summary["trips"].sum()
    * 100
)

display(
    destination_summary
    .sort_values("return_fare_rate")
    .round(2)
)

,drop_zone_type,trips,cancellations,cancellation_rate,return_fares,return_fare_rate,avg_distance_km,avg_fare_inr,share_of_trips
1,suburban,24643,5176,21.000,4073,16.530,23.380,353.570,41.070
2,tech_park,10075,903,8.960,4174,41.430,16.100,255.230,16.790
0,city_core,25282,2181,8.630,13329,52.720,13.240,216.620,42.140


In [ ]:
# B2.3 — Destination outcomes by airport terminal

terminal_destination = (
    trips
    .groupby(
        ["pickup_zone_id", "drop_zone_type"],
        as_index=False
    )
    .agg(
        trips=("trip_id", "count"),
        cancellation_rate=("captain_cancelled", "mean"),
        return_fare_rate=("got_return_fare_within_20min", "mean"),
        avg_distance_km=("trip_distance_km", "mean"),
        avg_fare_inr=("fare_inr", "mean")
    )
)

terminal_destination["cancellation_rate"] *= 100
terminal_destination["return_fare_rate"] *= 100

display(
    terminal_destination
    .sort_values(
        ["pickup_zone_id", "return_fare_rate"]
    )
    .round(2)
)

,pickup_zone_id,drop_zone_type,trips,cancellation_rate,return_fare_rate,avg_distance_km,avg_fare_inr
1,APT-T1,suburban,12350,21.070,16.490,23.360,353.490
2,APT-T1,tech_park,5078,8.780,41.140,16.060,254.520
0,APT-T1,city_core,12656,8.520,53.050,13.210,216.290
4,APT-T2,suburban,12293,20.940,16.570,23.390,353.640
5,APT-T2,tech_park,4997,9.150,41.730,16.150,255.950
3,APT-T2,city_core,12626,8.740,52.390,13.260,216.950


## B3 — Is Targeted Acquisition the Right Intervention?

### Decision

Targeted acquisition should not be the first intervention.

The airport supply shortage is highly concentrated by time and location rather than
being a system-wide shortage. Meanwhile, airport→suburban trips show substantially
lower return-fare availability and higher captain cancellation than other
destinations.

Therefore, the first intervention should target supply utilization and captain
economics in the specific shortage window and airport terminals. Acquisition should
be considered only if a residual supply gap remains after these interventions.

### Recommended intervention sequence

1. Test time-bound supply incentives for APT-T1/T2 during 21:00–02:00.
2. Test improved matching / return-trip opportunities for airport→suburban trips.
3. Measure incremental fulfillment, ETA, captain participation and unit economics.
4. If a persistent shortage remains, run targeted acquisition for captains able to
   serve the identified airport windows and locations.

### Limitation

The data does not contain individual captain IDs in `airport_trips.csv`, so we
cannot directly measure whether specific captains leave the airport after
suburban trips or whether poor return-fare availability causes that behavior.
The post-trip analysis is therefore directional rather than causal.

In [ ]:
# B3.1 — Quantify the intervention opportunity

# Core B1 concentration
shortage_hours = [21, 22, 23, 0, 1, 2]

shortage_requests = (
    hourly.loc[
        hourly["hour"].isin(shortage_hours),
        "requests"
    ].sum()
)

shortage_unfulfilled = (
    hourly.loc[
        hourly["hour"].isin(shortage_hours),
        "unfulfilled_requests"
    ].sum()
)

total_unfulfilled = hourly["unfulfilled_requests"].sum()

shortage_share = (
    shortage_unfulfilled / total_unfulfilled * 100
)

# Airport terminal concentration during shortage window
airport_shortage_unfulfilled = (
    zone_hourly[
        zone_hourly["zone_id"].isin(["APT-T1", "APT-T2"])
        & zone_hourly["hour"].isin(shortage_hours)
    ]["unfulfilled_requests"].sum()
)

airport_share = (
    airport_shortage_unfulfilled
    / shortage_unfulfilled
    * 100
)

# B2 suburban vs city-core return fare gap
destination = destination_summary.set_index("drop_zone_type")

suburban_return = destination.loc[
    "suburban", "return_fare_rate"
]

city_return = destination.loc[
    "city_core", "return_fare_rate"
]

return_fare_gap = city_return - suburban_return

suburban_cancel = destination.loc[
    "suburban", "cancellation_rate"
]

city_cancel = destination.loc[
    "city_core", "cancellation_rate"
]

cancellation_gap = suburban_cancel - city_cancel

print("B3 Intervention Opportunity")
print("-" * 45)
print(
    f"Unfulfilled requests in 21:00–02:00: "
    f"{shortage_unfulfilled:,.0f}"
)
print(
    f"Share of all unfulfilled requests: "
    f"{shortage_share:.1f}%"
)
print(
    f"APT-T1/T2 share of shortage-window unfulfilled: "
    f"{airport_share:.1f}%"
)

print("\nPost-trip destination signals")
print(
    f"Suburban return-fare rate: "
    f"{suburban_return:.1f}%"
)
print(
    f"City-core return-fare rate: "
    f"{city_return:.1f}%"
)
print(
    f"Return-fare gap: "
    f"{return_fare_gap:.1f} percentage points"
)
print(
    f"Suburban cancellation rate: "
    f"{suburban_cancel:.1f}%"
)
print(
    f"City-core cancellation rate: "
    f"{city_cancel:.1f}%"
)
print(
    f"Cancellation gap: "
    f"{cancellation_gap:.1f} percentage points"
)

B3 Intervention Opportunity
---------------------------------------------
Unfulfilled requests in 21:00–02:00: 45,091
Share of all unfulfilled requests: 70.1%
APT-T1/T2 share of shortage-window unfulfilled: 94.8%

Post-trip destination signals
Suburban return-fare rate: 16.5%
City-core return-fare rate: 52.7%
Return-fare gap: 36.2 percentage points
Suburban cancellation rate: 21.0%
City-core cancellation rate: 8.6%
Cancellation gap: 12.4 percentage points
